# Alfido Tech — Customer Behavior Analysis
Analyze customer transactions, RFM segments, purchase patterns, retention and churn-risk signals.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
DATA_PATH="ecommerce_customer_data_large.csv"
OUT=Path("alfido_customer_analysis"); (OUT/"visualizations").mkdir(parents=True,exist_ok=True); (OUT/"data").mkdir(parents=True,exist_ok=True)
df=pd.read_csv(DATA_PATH)
df["Purchase Date"]=pd.to_datetime(df["Purchase Date"],errors="coerce")
print(df.shape); display(df.head()); display(df.isna().sum())

In [ ]:
# Cleaning
df=df.drop_duplicates()
df["Returns"]=df["Returns"].fillna(0)
df=df.dropna(subset=["Customer ID","Purchase Date","Total Purchase Amount","Quantity"])
df=df[(df["Quantity"]>0)&(df["Total Purchase Amount"]>=0)&(df["Product Price"]>=0)]
print("Cleaned:",df.shape)

In [ ]:
# RFM feature engineering
snapshot=df["Purchase Date"].max()+pd.Timedelta(days=1)
rfm=df.groupby("Customer ID").agg(
 Recency=("Purchase Date",lambda x:(snapshot-x.max()).days),
 Frequency=("Purchase Date","count"),
 Monetary=("Total Purchase Amount","sum"),
 Avg_Order_Value=("Total Purchase Amount","mean"),
 Total_Quantity=("Quantity","sum"), Return_Rate=("Returns","mean"),
 Churn=("Churn","max"), Age=("Customer Age","first"), Gender=("Gender","first")).reset_index()
rfm["R_Score"]=pd.qcut(rfm.Recency.rank(method="first"),5,labels=[5,4,3,2,1]).astype(int)
rfm["F_Score"]=pd.qcut(rfm.Frequency.rank(method="first"),5,labels=[1,2,3,4,5]).astype(int)
rfm["M_Score"]=pd.qcut(rfm.Monetary.rank(method="first"),5,labels=[1,2,3,4,5]).astype(int)
rfm["RFM_Score"]=rfm[["R_Score","F_Score","M_Score"]].sum(axis=1)
def segment(s):
    return "Champions" if s>=13 else "Loyal / High Value" if s>=10 else "Potential Loyalists" if s>=7 else "At Risk" if s>=5 else "Hibernating"
rfm["Segment"]=rfm.RFM_Score.map(segment)
display(rfm.head())

In [ ]:
# Segment profile
profile=rfm.groupby("Segment").agg(Customers=("Customer ID","size"),Revenue=("Monetary","sum"),
Avg_Recency=("Recency","mean"),Avg_Frequency=("Frequency","mean"),Avg_Monetary=("Monetary","mean"),
Churn_Rate=("Churn","mean"),Return_Rate=("Return_Rate","mean")).reset_index()
profile["Revenue_Share"]=profile.Revenue/profile.Revenue.sum()
display(profile.sort_values("Revenue",ascending=False))

In [ ]:
# Purchase patterns
monthly=df.assign(Month=df["Purchase Date"].dt.to_period("M").astype(str)).groupby("Month").agg(
Transactions=("Customer ID","size"),Unique_Customers=("Customer ID","nunique"),Revenue=("Total Purchase Amount","sum")).reset_index()
category=df.groupby("Product Category").agg(Transactions=("Customer ID","size"),Customers=("Customer ID","nunique"),Revenue=("Total Purchase Amount","sum")).reset_index()
fig,ax=plt.subplots(); ax.plot(pd.to_datetime(monthly.Month),monthly.Revenue/1e6); ax.set(title="Monthly Revenue",ylabel="Revenue (million)"); plt.show()
fig,ax=plt.subplots(); ax.bar(category["Product Category"],category.Revenue/1e6); ax.set(title="Revenue by Category",ylabel="Revenue (million)"); plt.show()

In [ ]:
# Cohort retention
x=df[["Customer ID","Purchase Date"]].copy(); x["OrderMonth"]=x["Purchase Date"].dt.to_period("M")
first=x.groupby("Customer ID").OrderMonth.min().rename("CohortMonth"); x=x.join(first,on="Customer ID")
x["CohortIndex"]=(x.OrderMonth-x.CohortMonth).apply(lambda z:z.n)
cohort=x.groupby(["CohortMonth","CohortIndex"])["Customer ID"].nunique().reset_index()
cp=cohort.pivot(index="CohortMonth",columns="CohortIndex",values="Customer ID")
retention=cp.divide(cp.iloc[:,0],axis=0)
fig,ax=plt.subplots(figsize=(12,8)); im=ax.imshow(retention.iloc[:,:13],aspect="auto",vmin=0,vmax=1)
ax.set(title="Cohort Retention",xlabel="Months since first purchase",ylabel="Cohort month"); fig.colorbar(im,ax=ax,label="Retention"); plt.show()

## Recommendations
1. Champions/VIP journey with early access and loyalty rewards.
2. At-Risk win-back campaigns triggered by inactivity.
3. Low-cost Hibernating reactivation before deep discounts.
4. Category-based cross-selling and complementary recommendations.
5. Improve churn-target definition and early lifecycle/second-purchase measurement.

Export customer RFM, segment profile, purchase trends, category/payment summaries, cohort retention and churn diagnostics to CSV for GitHub/reporting.